# Foot detector â€” SynFoot 10K subset (Colab GPU, fresh account)

Fast/cheap pretrain for the **in-app Try-On AR path (NOT Snap)**.
Uses ~10K of the 43,742 train images via `fraction=0.229` (10,000 / 43,742).
Val still runs on the full 6,258 val split.
Expect ~6-12 min on a T4. Output contract is identical to 25k/50k:
1 class (foot) + 2 keypoints (heel, toe), ONNX opset=12 â€”
drop-in for `customer-portal/public/vendor/foot-detector/foot-v1.onnx`.

In [ ]:
# 0) Mount Drive â€” checkpoints get backed up here so a GPU-limit kill doesn't lose them
from google.colab import drive
drive.mount('/content/drive')
BACKUP_DIR = '/content/drive/MyDrive/foot-detector-train'
!mkdir -p "$BACKUP_DIR"
print(BACKUP_DIR)

In [ ]:
!nvidia-smi -L
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!pip install ultralytics onnx onnxruntime albumentations

In [ ]:
# Restore repo files from Drive if this is a fresh VM.
# Upload once: data-synfoot.yaml + synfoot-50k.zip under MyDrive/foot-detector-train/
import os
os.makedirs('training/foot-detector', exist_ok=True)
!cp -n "/content/drive/MyDrive/foot-detector-train/data-synfoot.yaml" training/foot-detector/ 2>/dev/null || echo "yaml already in repo, skipping cp"
!if [ ! -d training/foot-detector/datasets/synfoot/images/train ]; then unzip -q "/content/drive/MyDrive/foot-detector-train/synfoot-50k.zip" -d training/foot-detector/; fi
!python -c "import os; b='training/foot-detector/datasets/synfoot'; print({s: {k: len(os.listdir(os.path.join(b,k,s))) for k in ['images','labels']} for s in ['train','val']})"
# expect train 43742 / val 6258; fraction=0.229 below uses ~10K of train

In [ ]:
# Ultralytics resolves a relative `path:` against its own datasets folder,
# not against this yaml â€” rewrite it to the absolute VM path or train fails
# with a doubled `datasets/datasets/...` lookup. Idempotent: safe to re-run.
p = 'training/foot-detector/data-synfoot.yaml'
t = open(p).read()
if 'path: datasets/synfoot' in t:
    t = t.replace('path: datasets/synfoot', 'path: /content/training/foot-detector/datasets/synfoot')
    open(p, 'w').write(t)
print(open(p).read())

In [ ]:
# FRESH train â€” 10K subset for Try-On AR (not Snap). save_period=10 keeps epoch10/20/...
# so a quota kill only loses â‰¤10 epochs.
!yolo pose train \
  data=training/foot-detector/data-synfoot.yaml \
  model=yolo11n-pose.pt \
  epochs=100 imgsz=640 batch=16 fraction=0.229 save_period=10 \
  hsv_h=0.015 hsv_s=0.4 hsv_v=0.4 \
  degrees=15 translate=0.1 scale=0.3 \
  mosaic=0.5 mixup=0.1 copy_paste=0.1 seed=0 \
  project=training/foot-detector/runs name=synfoot-10k

In [ ]:
%%bash
BACKUP_DIR="/content/drive/MyDrive/foot-detector-train"
mkdir -p "$BACKUP_DIR"
SRC_DIR=$(find /content -type d -path "*synfoot-10k/weights" 2>/dev/null | head -n 1)
echo "weights: $SRC_DIR"
ls -lh "$SRC_DIR/" | head -20
cp "$SRC_DIR"/last.pt "$BACKUP_DIR/synfoot-10k-last.pt"
cp "$SRC_DIR"/best.pt "$BACKUP_DIR/synfoot-10k-best.pt"
ls -lh "$BACKUP_DIR/" | grep -E "10k|last|best"

In [ ]:
!yolo pose val data=training/foot-detector/data-synfoot.yaml model=training/foot-detector/runs/synfoot-10k/weights/best.pt

In [ ]:
# Export for Try-On AR browser path (opset 12 = onnxruntime-web coverage).
# Try-On AR uses this ONNX; Snap lens stays on its own tracking.
!yolo export model=training/foot-detector/runs/synfoot-10k/weights/best.pt format=onnx opset=12 dynamic=False
!ls -lh training/foot-detector/runs/synfoot-10k/weights/